# Automation for file Rearranging and Compiling

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
from utility import read_and_write


# Meta data reading

In [22]:
# Resovle script directory dynamically
# Assume the program is a python scirpt first, and then jupyter notebook if it fails (just for migration in the future)
try:
    path_script = Path(__file__).resolve().parent
except:
    path_script = Path.cwd()
path_script

intermediary = 'example_data'
path_data = path_script / intermediary

df_md = read_and_write.read_spectrum_meta_data(path_data, tools_to_read = ['H1', 'J4', 'K1', 'K4'])

print(df_md.info())
df_md.head()


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tool      4 non-null      string        
 1   wafer_id  4 non-null      string        
 2   datetime  4 non-null      datetime64[us]
 3   path      4 non-null      object        
 4   lot_id    4 non-null      string        
dtypes: datetime64[us](1), object(1), string(3)
memory usage: 292.0+ bytes
None


,tool,wafer_id,datetime,path,lot_id
0,H1,NTA000004.23,2026-05-17 15:26:21,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.0H
1,J4,NTA000001.16,2026-05-17 01:51:17,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.0K
2,K1,NTA000003.22,2026-05-21 13:03:47,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.0H
3,K4,NTA000002.19,2026-05-29 02:30:46,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.0H


### Merge with part

In [23]:
path_part = path_data/'part6.csv'
df_part = pd.read_csv(path_part)
df_part['lot'] = df_part['lot_id'].apply(lambda x: str(x).split('.')[0])
df_tmp = df_md.copy()
df_tmp['lot'] = df_tmp['wafer_id'].apply(lambda x: str(x).split('.')[0])

df_tmp = pd.merge(df_tmp, df_part, on='lot', suffixes=('', '_y'))
df_tmp.drop('lot_id_y', axis=1, inplace=True)
df_md = df_tmp
df_md.head()

,tool,wafer_id,datetime,path,lot_id,lot,part
0,H1,NTA000004.23,2026-05-17 15:26:21,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.0H,NTA000004,A12345
1,J4,NTA000001.16,2026-05-17 01:51:17,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.0K,NTA000001,A55555
2,K1,NTA000003.22,2026-05-21 13:03:47,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.0H,NTA000003,A12345
3,K4,NTA000002.19,2026-05-29 02:30:46,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.0H,NTA000002,A55555


In [27]:
df_tmp = df_md.copy()
df_tmp = df_tmp.sort_values('datetime') # merge_asof request sorted comparison key

time_tolerance_in_hours = tolerance=pd.Timedelta('1124h')

lst = []
for tool_source in df_tmp['tool'].unique():
    for tool_target in df_tmp['tool'].unique():
        if tool_source == tool_target:
            continue
        # specify source and target tool
        df_source = df_tmp.query('tool == @tool_source').copy()
        df_target = df_tmp.query('tool == @tool_target').copy()
        # duplicate datetime so it is not removed after merge_asof
        df_target['datetime_target'] = df_target['datetime']
        # apply merge_asof and calcualte time difference
        df_source = pd.merge_asof(df_source, df_target, on='datetime', by='part', direction='nearest', tolerance=time_tolerance_in_hours, suffixes=('','_target'))
        df_source['datetime_difference'] = df_source['datetime_target'] - df_source['datetime']

        lst.append(df_source)

df_md_paired = pd.concat(lst)#, columns=lst[0].columns)

df_md_paired.head()

,tool,wafer_id,datetime,path,lot_id,lot,part,path_spe,path_vms,path_spe_tranformed,...,wafer_id_target,path_target,lot_id_target,lot_target,path_spe_target,path_vms_target,path_spe_tranformed_target,path_spe_tranformed_lower_target,datetime_target,datetime_difference
0,J4,NTA000001.16,2026-05-17 01:51:17,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.0K,NTA000001,A55555,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,...,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT
0,J4,NTA000001.16,2026-05-17 01:51:17,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.0K,NTA000001,A55555,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,...,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT
0,J4,NTA000001.16,2026-05-17 01:51:17,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.0K,NTA000001,A55555,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,...,NTA000002.19,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.0H,NTA000002,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_sp...,2026-05-29 02:30:46,12 days 00:39:29
0,H1,NTA000004.23,2026-05-17 15:26:21,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.0H,NTA000004,A12345,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,...,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT
0,H1,NTA000004.23,2026-05-17 15:26:21,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.0H,NTA000004,A12345,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,...,NTA000003.22,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.0H,NTA000003,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_vm...,d:\gitlab\GGRD\Automation\example_data\data_sp...,d:\gitlab\GGRD\Automation\example_data\data_sp...,2026-05-21 13:03:47,3 days 21:37:26


# Manual VMS transformation

In [25]:
# standard usage:
# 1. set: perform_manual_tranform = True; dry_run = False
# 2. run the notebook, it will raise exception in this cell
# 3. do the spe to vms transformation by hand in the spe folder
# 4. run from this cell again, done
#
# omitting data copy but sitll getting path info stored in df_meat after the standard usage:
# 1. set: perform_manual_tranform = True; dry_run = True
# 2. the script will get you the vms/spe path without copying files
perform_manual_tranform = True
dry_run = False # you can use this to get related paths without wasting time on copying files

# copy and rename the spe files
if perform_manual_tranform:
    path_data_spe = path_data/'data_spe'
    df_md = read_and_write.copy_and_rename_spe(df_md, path_data_spe, dry_run)

# remind the user to do manual transform and copy vms files
if perform_manual_tranform:
    # raise exception to remind transformation
    if len(list(path_data_spe.glob('*.vms'))) == 0:
        print('No vms files found in the spe folder!')
        print('If this is the first run, do the tranform and run from this cell again.')
        raise Exception('No vms files found. Pls complete manual transformation and run again.')
    # copy the vms files
    path_data_vms = path_data/'data_vms'
    df_md = read_and_write.copy_and_rename_vms(df_md, path_data_vms, dry_run)
    df_md.to_csv(path_data/'df_meta_data.csv')

        


Missing 0 vms file(s)
Case used: Lower=False, Upper=True


In [26]:
# path_script = path_script / 'example_data' / 'H1'
# path_script
# file_path = Path(r'RawData_USERNAME_2026-05-29_09384401\NTA000004.0H\NTA000004.23_20260517-152621\G94.NXB083468.01.01.20260517.152441.spe'.replace('\\', '/'), )
# file_path = path_script / file_path
# file_path.parent.mkdir(parents=True, exist_ok=True)
# file_path.touch()
# file_path.parent